# 12 pamoka - Pokalbių istorijos sumažinimas naudojant agento užrašų bloknotą

Šis užrašų bloknotas iliustruoja, kaip valdyti kontekstą ilgose pokalbių eilutėse naudojant Microsoft agentų sistemą. Pokalbiams augant, padidėja žetonų skaičius — galiausiai viršijant modelio konteksto langą. Tai sprendžiame naudodami **konteksto santraukos modelį** ir **agentų užrašų bloknotą** nuolatinei atminčiai.

## Ko išmoksite:
1. **Kodėl svarbu valdyti kontekstą**: suprasti žetonų ribas ir konteksto langus
2. **Konteksto suvokiantys agentai**: kurti agentus, kurie valdo savo pokalbio kontekstą
3. **Konteksto santraukos modelis**: naudoti įrankius pokalbių istorijos sutrumpinimui
4. **Agentų užrašų bloknotas**: nuolatinė atmintis, kuri išlieka net sumažinus kontekstą

## Reikalingos žinios:
- Azure OpenAI nustatymas su aplinkos kintamaisiais
- Pagrindinių agentų koncepcijų supratimas iš ankstesnių pamokų


## Diegimas


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Kodėl svarbu valdyti kontekstą

Kiekvienas didelis kalbos modelis (LLM) turi ribotą **konteksto langą** — maksimalų žetonų skaičių, kurį jis gali apdoroti viename užklausoje. Vykstant daugiaruniui pokalbiui:

- **Žetonų skaičius auga linijiniu būdu** su kiekvienu vartotojo pranešimu ir asistento atsakymu.
- **Klausimo žetonai lemia kainą**, nes visa istorija siunčiama iš naujo kiekviename žingsnyje.
- Galiausiai pokalbis **viršija konteksto langą** ir modelis arba apkarpo tekstą, arba įvyksta klaida.

### Konteksto valdymo strategijos

| Strategija | Kaip veikia | Pasirinkimo kaina |
|---|---|---|
| **Apkarpymas** | Pašalinti seniausius pranešimus | Prarandamas ankstesnis kontekstas |
| **Santrauka** | Suvesti senesnius pranešimus į santrauką | Prarandama dalis detalių, bet išlaikomi pagrindiniai punktai |
| **Užrašų / išorinės atminties naudojimas** | Saugoti svarbią informaciją už pokalbio ribų | Reikia įrankių iškvietimų, bet išlaiko informaciją net ir mažinant pokalbį |

Šiame užrašų knygyje mes deriname **santraukų rengimą** su **užrašų įrankiu**, kad agentas galėtų palaikyti tęstinumą net ir sutrumpinus pokalbio istoriją.


## Kurti kontekstą suprantantį agentą


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Ilgos pokalbio simuliacija

Pažvelkime į daugiapakopį pokalbį, kad pamatytume, kaip kaupiasi kontekstas. Agentas turėtų išlaikyti svarbias detales (preferencijas, biudžetą, kelionės datas) per visus posūkius ir parodyti tęstinumą.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Atkreipkite dėmesį, kaip agentas išlaiko konteksą iš ankstesnių pokalbio etapų — jis žino apie Japoniją, sushi, šventyklas, fotografiją, 3000 USD biudžetą, keliones vienam ir balandžio laikotarpį. Trumpame pokalbyje tai veikia gerai, tačiau, kai pokalbis auga, visa istorija tampa per brangi ir ją vėl siųsti.

Tęskime pokalbį su daugiau etapų, kad pamatytume konteksto kaupimą:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Konteksto santraukos šablonas

Pokalbiui augant, galime naudoti **santraukos įrankį**, kad sutrauktume sukauptą kontekstą į kompaktišką formatą. Agentas kviečia šį įrankį užrašyti pagrindines nuostatas, kad net jei senesnės žinutės būtų pašalintos, svarbi informacija būtų išsaugota.

Šis šablonas yra pamatas kurti sudėtingesniems istorijos sutrumpinimams:
1. Agentas identifikuoja pagrindinius faktus iš pokalbio
2. Jis kviečia santraukos įrankį juos išsaugoti
3. Senesnės žinutės gali būti saugiai pašalintos, nes santrauka apima svarbiausią informaciją

Žemiau apibrėžiame `summarize_preferences` įrankį, kurį agentas gali kviečiant įrašyti kompaktišką apibendrinimą to, ką jis sužinojo.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Santrauka

Šioje pamokoje sužinojote, kaip valdyti kontekstą ilgai trunkančiose agentų pokalbiuose, naudojant Microsoft Agent Framework:

### Pagrindinės sąvokos
- **Konteksto langai yra riboti** — kiekvienas pokalbio istorijos žodis kainuoja ir skaičiuojamas į ribą.
- **Santraukos įrankiai** leidžia agentui suspausti sukauptą kontekstą į trumpas santraukas, sumažinant žodžių naudojimą ir išlaikant svarbią informaciją.
- **Agentų užrašų bloknotai** suteikia nuolatinę išorinę atmintį, kuri išlieka nepaisant pokalbio sutrumpinimo.

### Ką Jūs Sukūrėte
- **Konteksto suvokiantį agentą**, kuris išlaiko tęstinumą daugiaaukščiuose pokalbiuose
- **Santraukos įrankį** (`summarize_preferences`), kuris įrašo pagrindinius naudotojo duomenis kompaktišku formatu
- **Daugiaaukštį pokalbį**, demonstruojantį konteksto išlaikymą ir pokyčių valdymą

### Tikro Pasaulio Taikymai
- **Klientų aptarnavimo robotai**: prisimena pageidavimus per ilgus palaikymo sesijų laikotarpius
- **Asmeniniai asistentai**: stebi vykstančius projektus be būtinybės iš naujo aiškinti kontekstą
- **Mokomieji mokytojai**: išlaiko mokinio pažangą per daug sąveikų

### Tolimesni žingsniai
- Įdiegti pilną užrašų bloknotų įrankį su failų pagrindu išlaikoma atmintimi
- Pridėti automatinį istorijos sutrumpinimą po santraukos sudarymo
- Sujungti su vektorinėmis duomenų bazėmis semantinės atminties paieškai
- Kurti agentus, galinčius pratęsti pokalbius po kelių dienų turėdami pilną kontekstą


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
